In [1]:
import os
import numpy as np
from dwave.system import LeapHybridNLSampler
from dwave.optimization.model import Model
from dotenv import load_dotenv, find_dotenv
import networkx as nx

In [2]:
# Load Environmental Variables
load_dotenv(find_dotenv())
token = os.environ['DWAVE_API_KEY'] 

In [3]:
# Define the number of facilities/locations
n = 4

# Define the flow matric
f = np.array([[0, 3, 0, 2], [3, 0, 0, 1], [0, 0, 0, 4], [2, 1, 4, 0]])

# Define the distance matrix
d = np.array([[0, 22, 53, 53], [22, 0, 40, 62], [53, 40, 0, 55], [53, 62, 55, 0]])

offset = 0.5

### Let's compare the case for n=9

In [4]:
# Define the number of facilities/locations
n = 9

# Define the flow matrix
f = np.array([[0, 2, 4, 0, 0, 0, 2, 0, 0], [2, 0, 3, 1, 0, 6, 0, 0, 2], [4, 3, 0, 0, 0, 3, 0, 0, 0],
              [0, 1, 0, 0, 1, 0, 1, 2, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 6, 3, 0, 0, 0, 0, 0, 2],
              [2, 0, 0, 1, 0, 0, 0, 4, 3], [0, 0, 0, 2, 0, 0, 4, 0, 0], [0, 2, 0, 0, 0, 2, 3, 0, 0]])

# Define the distance matrix
d = np.array([[0, 32, 68, 97, 75, 70, 75, 40, 24], [32, 0, 42, 80, 53, 65, 82, 47, 29], [68, 42, 0, 45, 15, 49, 79, 55, 50],
              [97, 80, 45, 0, 30, 36, 65, 65, 73], [75, 53, 15, 30, 0, 38, 69, 53, 53], [70, 65, 49, 36, 38, 0, 31, 32, 46],
              [75, 82, 79, 65, 69, 31, 0, 36, 56], [40, 47, 55, 65, 53, 32, 36, 0, 19], [24, 29, 50, 73, 53, 46, 56, 19, 0]])


In [5]:
def build_NLM():

    # Construct the model
    qap_model = Model()

    # Represent the facility assignments efficiently using permutations of an ordered list
    num_facilities = d.shape[0]
    ordered_facilities = qap_model.list(num_facilities)

    # Add the constants
    DISTANCE_MATRIX = qap_model.constant(d)
    FLOW_MATRIX = qap_model.constant(f)
    
    # Minimize the sum of distances multiplied by the flows.
    qap_model.minimize(
        (
            DISTANCE_MATRIX * FLOW_MATRIX[ordered_facilities, :][:, ordered_facilities]
        ).sum()
                    )
    qap_model.lock()

    return qap_model

In [6]:
def solve(NL_model, time_limit=5):

  """
    Solve the Non-Linear Model (NL) using the LeapHybridNLSampler.

    Args:NL Model to solve.
        time_limit (int): Time limit in seconds for the solver.

    Returns:
        sampleset (SampleSet): The resulting sampleset from the solver.
    """

  sampler = LeapHybridNLSampler(token=token)
  results = sampler.sample(
                            NL_model,
                            time_limit=time_limit,
                            label='QAP_NL')
  route, = NL_model.iter_decisions()
  
  feasible_samples = []
  energy = []
  for i in range(NL_model.states.size()):
      if not (NL_model.iter_constraints()):
          raise ValueError("No feasible solution found")
      else:
          feasible_samples.append((route.state(i)))
          energy.append(NL_model.objective.state(i))
  best = feasible_samples[0]
  minimum_objective = energy[0]

  return best, minimum_objective

In [7]:
qap_model = build_NLM()

In [8]:
sampleset, energy = solve(qap_model)

In [9]:
print(f'Facility_sequence according to the order of location[0-indexed]: {sampleset}')

Facility_sequence according to the order of location[0-indexed]: [2. 0. 8. 7. 6. 3. 4. 5. 1.]


In [10]:
print(f'Total Cost: {energy*offset}')

Total Cost: 1160.0
